# Volatility research notebook: does selling premium pay, and which signals matter?

Tests ideas from the Roman Paolucci videos (volatility risk premium, trading IV, ARCH/GARCH, volatility drag) on **Unusual Whales' 2-year daily implied/realized volatility history**, for the use case of **defined-risk put credit spreads in a small account**.

| Test | Video idea | Question |
|---|---|---|
| 1 | Volatility risk premium | Is implied vol higher than the vol that followed? Is the regression slope below 1? |
| 2 | ARCH/GARCH, window choice | Which forecast of next month's realized vol is best: trailing 21d, 63d, 252d, the blend, EWMA, GARCH, or IV itself? |
| 3 | Statistical basis for a trade | Do the screener's signals (IV/HV bands, IV rank) actually sort outcomes? |
| 4 | Trading implied vol | Does IV mean-revert, and does the crossover level hold **out of sample**? |
| 5 | Credit spreads | How often is the short strike breached versus its delta, and what would the spread have earned? |
| 6 | Fat tails | Do normal-distribution risk numbers understate losses? |
| 7 | Volatility drag / sizing | Kelly fraction, and what bet size survives, for a $3,000 account |

**Run it:** `pip install pandas numpy scipy statsmodels arch matplotlib`, set `UW_API_KEY`, run all cells. With no key it runs on **synthetic data** (clearly labelled) so you can check the code; do not draw conclusions from that.

**Read this before believing any number.** Two years is a short sample and 21-day windows overlap, so the effective sample is small; the code reports block-bootstrap intervals for that reason. Spread P&L here is a **model** (Black-Scholes at one flat IV, held to expiry, no skew, no early exit), not real option prices. Treat results as "is there any signal worth testing properly", not as a backtest you can trade.

In [ ]:
import os, json, math, time, warnings, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import statsmodels.api as sm
try:
    from arch import arch_model
    HAVE_ARCH = True
except Exception:
    HAVE_ARCH = False
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40); pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": .25})

# ---------------- settings you may change ----------------
UNIVERSE = ["SPY", "QQQ", "IWM", "XLF", "XLE", "GLD", "TLT", "JPM", "KO", "AAPL", "MU", "NVDA"]
DATA_MODE = "auto"            # "auto": live if UW_API_KEY is set, else synthetic | "live" | "synthetic"
CACHE_DIR = Path("data_uw")   # live responses are cached here (delete the folder to refresh)
HORIZON = 21                  # trading days in UW's realized-vol window (30 calendar days of IV)
DTE_DAYS = 30
RF = 0.04                     # risk-free rate for Black-Scholes
TARGET_DELTA = 0.16           # short strike delta
ENTRY_STEP = 5                # simulate an entry every N trading days per ticker
FEE_PER_LEG = 1.00            # $ per contract per leg to OPEN (check tastytrade's current schedule)
SLIPPAGE = 0.02               # $ per share given up versus mid when opening the spread
ACCOUNT = 3000.0
MAX_RISK_PCT = 0.10           # a 1-lot should not risk more than this share of the account
RICH_AT, THIN_BELOW = 1.20, 0.95          # the screener's IV/HV bands (untested defaults)
IVR_HI, IVR_LO = 50, 25                   # IV rank conventions
FW = {21: 0.5, 63: 0.3, 252: 0.2}         # the screener's realized-vol blend (untested weights)
SEED = 7
print("arch available:", HAVE_ARCH)

## Data
Unusual Whales `GET /api/stock/{ticker}/volatility/realized?timeframe=2Y&days=21` returns one row per trading day: close price, 30-day implied vol, and the **21-day realized vol that followed** (null for the newest rows, whose window has not finished).

In [ ]:
BASE = "https://api.unusualwhales.com"

def fetch_uw_realized(ticker, key, timeframe="2Y", days=21):
    url = f"{BASE}/api/stock/{ticker}/volatility/realized?timeframe={timeframe}&days={days}"
    req = urllib.request.Request(url, headers={"Authorization": "Bearer " + key, "Accept": "application/json",
                                               "User-Agent": "screener-research/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        j = json.loads(r.read().decode("utf-8", "replace"))
    return j.get("data") if isinstance(j, dict) else j

def tidy(rows, ticker):
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None).dt.normalize()
    for c in ("price", "implied_volatility", "realized_volatility"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    df["ticker"] = ticker
    return df[["date", "ticker", "price", "implied_volatility", "realized_volatility"]].reset_index(drop=True)

def load_live(tickers, key):
    CACHE_DIR.mkdir(exist_ok=True)
    frames, failed = [], []
    for t in tickers:
        f = CACHE_DIR / f"{t}.csv"
        if f.exists() and time.time() - f.stat().st_mtime < 20 * 3600:
            frames.append(pd.read_csv(f, parse_dates=["date"])); continue
        try:
            rows = fetch_uw_realized(t, key)
            df = tidy(rows, t); df.to_csv(f, index=False); frames.append(df); time.sleep(0.3)
        except urllib.error.HTTPError as e:
            failed.append((t, e.code))
            if e.code == 401: raise SystemExit("401: Unusual Whales rejected the key. Check UW_API_KEY.")
        except Exception as e:
            failed.append((t, type(e).__name__))
    if failed: print("skipped:", failed)
    return pd.concat(frames, ignore_index=True)

def synth_ticker(ticker, seed, n=501):
    """Synthetic stand-in with a built-in 10% IV premium and mean-reverting vol. For testing the code only."""
    rng = np.random.RandomState(seed)
    mu_lv = np.log(rng.uniform(0.16, 0.38)); kappa, xi = 0.03, 0.09
    m = n + 80
    lv = np.empty(m); lv[0] = mu_lv
    for i in range(1, m): lv[i] = lv[i-1] + kappa * (mu_lv - lv[i-1]) + xi * rng.randn()
    vol = np.exp(lv)
    z = rng.standard_t(5, size=m) / np.sqrt(5 / 3)
    r = vol / np.sqrt(252) * z + 0.0003
    price = 100 * np.exp(np.cumsum(r))
    iv = (0.6 * vol + 0.4 * np.exp(mu_lv)) * 1.10 * np.exp(0.05 * rng.randn(m))
    rs = pd.Series(r)
    fwd = rs.rolling(HORIZON).std().shift(-HORIZON) * np.sqrt(252)      # r[t+1..t+21]
    dates = pd.bdate_range(end="2026-10-23", periods=m)
    df = pd.DataFrame({"date": dates, "ticker": ticker, "price": price, "implied_volatility": iv, "realized_volatility": fwd.values})
    return df.iloc[40:40 + n].reset_index(drop=True)

key = os.environ.get("UW_API_KEY", "").strip()
mode = DATA_MODE if DATA_MODE != "auto" else ("live" if key else "synthetic")
if mode == "live":
    raw = load_live(UNIVERSE, key)
else:
    raw = pd.concat([synth_ticker(t, SEED + i) for i, t in enumerate(UNIVERSE)], ignore_index=True)
    print("=" * 78, "\nSYNTHETIC DATA: this only checks that the code runs. Do not draw conclusions.\n" + "=" * 78)
IS_SYNTH = (mode != "live")
print(mode, "| tickers:", raw.ticker.nunique(), "| rows:", len(raw), "|", raw.date.min().date(), "to", raw.date.max().date())
raw.groupby("ticker").agg(rows=("date", "size"), first=("date", "min"), last=("date", "max"),
                          null_rv=("realized_volatility", lambda s: s.isna().sum()), iv_mean=("implied_volatility", "mean"))

### Check how UW defines the realized-vol column
If `realized_volatility` on row *t* is the vol of the **next** 21 daily returns, it should line up best with my own forward calculation at one offset `a`. This also catches annualization or unit differences.

In [ ]:
def own_fwd_rv(g, a):
    r = np.log(g["price"]).diff()
    return r.shift(-a).rolling(HORIZON).std().shift(-(HORIZON - 1)) * np.sqrt(252)

rows = []
for t, g in raw.groupby("ticker"):
    g = g.sort_values("date").reset_index(drop=True)
    for a in (0, 1, 2):
        own = own_fwd_rv(g, a)
        m = g["realized_volatility"].notna() & own.notna()
        if m.sum() > 50:
            rows.append(dict(ticker=t, offset=a, corr=g.loc[m, "realized_volatility"].corr(own[m]),
                             uw_over_own=(g.loc[m, "realized_volatility"] / own[m]).median()))
chk = pd.DataFrame(rows)
print(chk.groupby("offset")[["corr", "uw_over_own"]].mean())
best_a = int(chk.groupby("offset")["corr"].mean().idxmax())
print("best offset:", best_a, "(1 means the window starts the day AFTER the row date; uw_over_own near 1.0 means same units)")
gaps = raw.sort_values(["ticker", "date"]).groupby("ticker")["date"].diff().dt.days
print("largest gap between rows (days):", int(gaps.max()), "(over ~5 means missing sessions, which would misalign the 21-row windows)")

In [ ]:
def garch_forecast_series(ret, first=252, refit_every=21, horizon=HORIZON):
    """Annualized GARCH(1,1)-t forecast of the average vol over the next `horizon` days, using only data up to t.
    Refit every `refit_every` days; between refits the variance is updated with fixed parameters."""
    out = pd.Series(np.nan, index=ret.index)
    if not HAVE_ARCH or ret.notna().sum() < first + 30: return out
    rp = (ret * 100).to_numpy(); n = len(rp)
    params, sigma2, next_try = None, None, first
    for t in range(first, n):
        fresh = False
        if (params is None and t >= next_try) or (params is not None and (t - first) % refit_every == 0):
            try:
                res = arch_model(rp[1:t + 1], vol="GARCH", p=1, q=1, dist="t", mean="Constant", rescale=False).fit(disp="off", show_warning=False)
                p = res.params
                mu, om, al, be = float(p["mu"]), float(p["omega"]), float(p["alpha[1]"]), float(p["beta[1]"])
                if not (om > 0 and al >= 0 and be >= 0 and al + be < 0.999): raise ValueError("non-stationary")
                params, sigma2, fresh = (mu, om, al, be), float(np.asarray(res.conditional_volatility)[-1] ** 2), True
            except Exception:
                params, next_try = None, t + refit_every
                continue
        if params is None: continue
        mu, om, al, be = params
        if not fresh: sigma2 = om + al * (rp[t - 1] - mu) ** 2 + be * sigma2
        s_next = om + al * (rp[t] - mu) ** 2 + be * sigma2
        V, phi = om / (1 - al - be), al + be
        avg = V + (s_next - V) * (1 - phi ** horizon) / ((1 - phi) * horizon)
        out.iloc[t] = math.sqrt(max(avg, 1e-12) / 1e4 * 252)
    return out

def build_features(df):
    out = []
    ann = np.sqrt(252)
    for t, g in df.groupby("ticker", sort=False):
        g = g.sort_values("date").reset_index(drop=True).copy()
        g["ret"] = np.log(g["price"]).diff()
        g["rv21"] = g["ret"].rolling(21).std() * ann
        g["rv63"] = g["ret"].rolling(63).std() * ann
        g["rv252"] = g["ret"].rolling(252, min_periods=200).std() * ann
        g["blend"] = FW[21] * g["rv21"] + FW[63] * g["rv63"] + FW[252] * g["rv252"]
        g["ewma"] = np.sqrt((g["ret"] ** 2).ewm(alpha=0.06, adjust=False).mean() * 252)
        g.loc[:20, "ewma"] = np.nan
        g["garch"] = garch_forecast_series(g["ret"])
        g["iv"] = g["implied_volatility"]
        g["fwd_rv"] = g["realized_volatility"]
        g["fwd_iv_avg"] = g["iv"].rolling(HORIZON).mean().shift(-HORIZON)       # mean of iv[t+1..t+21]
        lo, hi = g["iv"].rolling(252, min_periods=126).min(), g["iv"].rolling(252, min_periods=126).max()
        g["ivr"] = 100 * (g["iv"] - lo) / (hi - lo)
        g["prem"] = 100 * (g["iv"] - g["fwd_rv"])                                  # vol points the market overpaid
        out.append(g)
    return pd.concat(out, ignore_index=True)

t0 = time.time()
F = build_features(raw)
print("features built in %.0fs" % (time.time() - t0), "| rows with a finished forward RV:", int(F.fwd_rv.notna().sum()))
SUMMARY = {}

def ols(y, x, groups=None):
    X = sm.add_constant(pd.Series(x).reset_index(drop=True)); yy = pd.Series(y).reset_index(drop=True)
    m = sm.OLS(yy, X)
    if groups is not None:
        return m.fit(cov_type="cluster", cov_kwds={"groups": np.asarray(groups)})
    return m.fit(cov_type="HAC", cov_kwds={"maxlags": HORIZON})

def mbb_mean_ci(groups, block=HORIZON, n_boot=1000, seed=0):
    """Moving-block bootstrap of the pooled mean. Overlapping windows make ordinary t-tests far too confident."""
    rng = np.random.RandomState(seed)
    groups = [np.asarray(g, float) for g in groups if len(g) > 0]
    if not groups: return (np.nan, np.nan, np.nan)
    point = np.concatenate(groups).mean(); boots = []
    for _ in range(n_boot):
        vals = []
        for g in groups:
            m = len(g); b = max(1, min(block, m)); k = int(np.ceil(m / b))
            starts = rng.randint(0, m - b + 1, size=k)
            vals.append(np.concatenate([g[s:s + b] for s in starts])[:m])
        boots.append(np.concatenate(vals).mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi

## Test 1: Is implied vol overpriced? (volatility risk premium)
The video regresses the vol that followed on today's IV. If IV were an unbiased forecast the slope would be 1 and the points would sit on y = x. **Slope below 1** and **mean premium above 0** is the evidence for selling premium. Roman also says the effect is stronger in high-vol regimes.

What to look for: the sign and size of the mean premium *with its bootstrap interval*, how many tickers show it, and whether the pooled slope's confidence interval excludes 1.

In [ ]:
d1 = F.dropna(subset=["iv", "fwd_rv"]).copy()
g_all = pd.factorize(d1.ticker)[0]
pooled = ols(d1.fwd_rv, d1.iv, g_all)
b, a = pooled.params.iloc[1], pooled.params.iloc[0]
ci = pooled.conf_int().iloc[1]
print(f"Pooled: fwd_rv = {a:.3f} + {b:.3f} * iv   (slope 95% CI {ci[0]:.2f} to {ci[1]:.2f}; clustered by ticker)   R2 = {pooled.rsquared:.2f}")
print("crossover (where IV equals the vol that followed):", "%.1f%%" % (100 * a / (1 - b)) if b < 1 else "none (slope >= 1)")

per = []
for t, g in d1.groupby("ticker"):
    r = ols(g.fwd_rv, g.iv)
    per.append(dict(ticker=t, n=len(g), slope=r.params.iloc[1], slope_se=r.bse.iloc[1], mean_iv=100 * g.iv.mean(), mean_fwd_rv=100 * g.fwd_rv.mean(),
                    mean_prem_pts=g.prem.mean(), pct_days_iv_above_rv=100 * (g.iv > g.fwd_rv).mean()))
per = pd.DataFrame(per).set_index("ticker")
display(per)

pt, lo, hi = mbb_mean_ci([g.prem.to_numpy() for _, g in d1.groupby("ticker")])
print(f"Mean premium (IV minus the vol that followed): {pt:.2f} vol points, 95% block-bootstrap CI {lo:.2f} to {hi:.2f}")
print(f"Share of days IV exceeded the realized vol that followed: {100 * (d1.iv > d1.fwd_rv).mean():.0f}%   | tickers with a positive mean premium: {(per.mean_prem_pts > 0).sum()} of {len(per)}")
SUMMARY["VRP mean premium (pts)"] = f"{pt:.2f} [{lo:.2f}, {hi:.2f}]"; SUMMARY["VRP pooled slope"] = f"{b:.2f} (CI {ci[0]:.2f} to {ci[1]:.2f})"

# regimes: high vs low IV relative to each ticker's own median
d1["regime"] = np.where(d1.iv > d1.groupby("ticker").iv.transform("median"), "high-IV", "low-IV")
for rg, g in d1.groupby("regime"):
    r = ols(g.fwd_rv, g.iv, pd.factorize(g.ticker)[0])
    print(f"{rg:8s} slope {r.params.iloc[1]:.2f} (se {r.bse.iloc[1]:.2f})  mean premium {g.prem.mean():.2f} pts  n={len(g)}")

fig, ax = plt.subplots()
ax.scatter(100 * d1.iv, 100 * d1.fwd_rv, s=4, alpha=.25)
xs = np.linspace(100 * d1.iv.min(), 100 * d1.iv.max(), 50)
ax.plot(xs, xs, "k--", label="y = x (IV unbiased)"); ax.plot(xs, a * 100 + b * xs, "r", label=f"fit, slope {b:.2f}")
ax.set_xlabel("implied vol today (%)"); ax.set_ylabel("realized vol over the next 21 days (%)"); ax.legend(); ax.set_title("Test 1: implied vs the vol that followed"); plt.show()

## Test 2: Which forecast of next month's realized vol is best?
This decides the denominator of IV/HV. Candidates: trailing 21d, 63d, 252d, the screener's 0.5/0.3/0.2 blend, EWMA (lambda 0.94), a rolling GARCH(1,1), and IV itself (plus IV scaled by its average ratio). All are scored on the **same rows** against the vol that actually followed.

Metrics: MAE and bias in vol points; **QLIKE** (lower is better; standard for volatility forecasts and not dominated by spikes); Mincer-Zarnowitz R2 (how much of the variation the forecast explains). The 252-day and GARCH forecasts need a year of history, so only the second year is scored, which is a small sample: read differences between close competitors as ties.

In [ ]:
F["iv_scaled"] = F["iv"] * (F["fwd_rv"] / F["iv"]).mean()
PRED = ["iv", "iv_scaled", "rv21", "rv63", "rv252", "blend", "ewma"] + (["garch"] if HAVE_ARCH else [])
ev = F.dropna(subset=["fwd_rv"] + PRED).copy()
print("scored rows:", len(ev), "| tickers:", ev.ticker.nunique(), "|", ev.date.min().date(), "to", ev.date.max().date())
qlike = lambda y, f: (lambda r: r - np.log(r) - 1)((y / f) ** 2)
grp = pd.factorize(ev.ticker)[0]
rows = []
for c in PRED:
    e = ev[c] - ev.fwd_rv
    mz = ols(ev.fwd_rv, ev[c], grp)
    rows.append(dict(predictor=c, MAE_pts=100 * e.abs().mean(), bias_pts=100 * e.mean(), QLIKE=qlike(ev.fwd_rv, ev[c]).mean(),
                     MZ_R2=mz.rsquared, MZ_slope=mz.params.iloc[1]))
res2 = pd.DataFrame(rows).set_index("predictor")
ref = ev.assign(l=qlike(ev.fwd_rv, ev["rv21"])).groupby("ticker").l.mean()
res2["tickers_beating_rv21"] = [int((ev.assign(l=qlike(ev.fwd_rv, ev[c])).groupby("ticker").l.mean() < ref).sum()) for c in res2.index]
display(res2.sort_values("QLIKE"))
best = res2.QLIKE.idxmin()
print("lowest QLIKE:", best, "| MZ_slope near 1 means the forecast is well scaled; below 1 means it overreacts to its own recent moves")
SUMMARY["Best realized-vol forecast (QLIKE)"] = best
res2.QLIKE.sort_values().plot.barh(title="Test 2: forecast loss (QLIKE, lower is better)"); plt.show()

## Test 3: Do the screener's signals sort outcomes?
For each day, compute the IV/HV ratio (using each candidate denominator) and the IV rank, bucket them with the screener's cut-offs, and look at what happened next: the **premium that was actually captured** (IV minus the vol that followed). If "rich" days do not show a bigger premium than "thin" days, the label adds nothing. No thresholds are tuned here.

In [ ]:
def bucket_rows(d, masks, value="prem"):
    out = []
    for name, m in masks.items():
        s = d[m & d[value].notna()]
        groups = [g[value].to_numpy() for _, g in s.groupby("ticker")]
        pt, lo, hi = mbb_mean_ci(groups, block=HORIZON, n_boot=600)
        out.append(dict(bucket=name, n=len(s), tickers=s.ticker.nunique(), mean_prem_pts=pt, ci_lo=lo, ci_hi=hi,
                        pct_iv_above_rv=100 * (s.iv > s.fwd_rv).mean() if len(s) else np.nan))
    return pd.DataFrame(out).set_index("bucket")

base = F.dropna(subset=["iv", "fwd_rv"]).copy()
for den in ["rv21", "blend"] + (["garch"] if HAVE_ARCH else []):
    d = base.dropna(subset=[den]).copy(); d["ratio"] = d["iv"] / d[den]
    print(f"\n--- IV / {den}   (rows {len(d)})")
    display(bucket_rows(d, {"thin  (<%.2f)" % THIN_BELOW: d.ratio < THIN_BELOW,
                            "fair": (d.ratio >= THIN_BELOW) & (d.ratio < RICH_AT),
                            "rich  (>=%.2f)" % RICH_AT: d.ratio >= RICH_AT}))

d = base.dropna(subset=["ivr", "rv21"]).copy(); d["ratio"] = d["iv"] / d["rv21"]
print("\n--- IV rank buckets (rank needs a year of history, so only the second year is used)")
display(bucket_rows(d, {"IV rank < %d" % IVR_LO: d.ivr < IVR_LO, "mid": (d.ivr >= IVR_LO) & (d.ivr < IVR_HI), "IV rank >= %d" % IVR_HI: d.ivr >= IVR_HI}))
print("\n--- Both signals together (IV/HV21 rich vs not, IV rank high vs not)")
rich, hi_r = d.ratio >= RICH_AT, d.ivr >= IVR_HI
tab = bucket_rows(d, {"rich & high rank": rich & hi_r, "rich & low/mid rank (the JPM case)": rich & ~hi_r, "not rich & high rank": ~rich & hi_r, "not rich & low/mid rank": ~rich & ~hi_r})
display(tab)
SUMMARY["Premium: rich&high-rank vs not-rich&low-rank"] = f"{tab.iloc[0].mean_prem_pts:.2f} vs {tab.iloc[3].mean_prem_pts:.2f} pts"

## Test 4: Does implied vol mean-revert? (the video's crossover)
Regress the **average IV over the next 21 days** on today's IV, per ticker. A slope below 1 means IV drifts back toward a stock-specific level; the crossover (a / (1 - b)) is that level. The video finds it in-sample on one stock (NVDA, 64%). Here it is **estimated on the first half of each ticker's history and tested on the second half**: when IV is above the training crossover, did it actually fall afterward? Also: below the crossover, did it rise?

In [ ]:
d4 = F.dropna(subset=["iv", "fwd_iv_avg"]).copy()
rows, test_parts = [], []
for t, g in d4.groupby("ticker"):
    g = g.sort_values("date"); k = len(g) // 2
    tr, te = g.iloc[:k], g.iloc[k:].copy()
    r = ols(tr.fwd_iv_avg, tr.iv); a_, b_ = r.params.iloc[0], r.params.iloc[1]
    cross = a_ / (1 - b_) if 0 < b_ < 1 else np.nan
    rows.append(dict(ticker=t, train_slope=b_, train_crossover_pct=100 * cross if cross == cross else np.nan,
                     train_mean_iv=100 * tr.iv.mean(), test_mean_iv=100 * te.iv.mean()))
    if cross == cross:
        te["above"] = te.iv > cross; te["d_iv"] = 100 * (te.fwd_iv_avg - te.iv); test_parts.append(te)
tr_tab = pd.DataFrame(rows).set_index("ticker"); display(tr_tab)
if test_parts:
    T = pd.concat(test_parts)
    res4 = []
    for name, m in {"IV above training crossover": T.above, "IV below training crossover": ~T.above}.items():
        s = T[m]
        pt, lo, hi = mbb_mean_ci([g.d_iv.to_numpy() for _, g in s.groupby("ticker")], n_boot=600)
        res4.append(dict(bucket=name, n=len(s), mean_fwd_change_pts=pt, ci_lo=lo, ci_hi=hi, pct_that_fell=100 * (s.d_iv < 0).mean()))
    res4 = pd.DataFrame(res4).set_index("bucket"); display(res4)
    SUMMARY["IV out-of-sample change when above/below crossover (pts)"] = " / ".join(f"{v:.2f}" for v in res4.mean_fwd_change_pts)
    print("Expectation if mean reversion holds out of sample: negative when above, positive when below. Intervals that include 0 mean 'not shown'.")
fig, ax = plt.subplots(); ax.scatter(100 * d4.iv, 100 * d4.fwd_iv_avg, s=4, alpha=.25)
xs = np.linspace(100 * d4.iv.min(), 100 * d4.iv.max(), 50); ax.plot(xs, xs, "k--", label="y = x")
r = ols(d4.fwd_iv_avg, d4.iv, pd.factorize(d4.ticker)[0]); ax.plot(xs, 100 * r.params.iloc[0] + r.params.iloc[1] * xs, "r", label=f"pooled fit, slope {r.params.iloc[1]:.2f}")
ax.set_xlabel("IV today (%)"); ax.set_ylabel("average IV over the next 21 days (%)"); ax.legend(); ax.set_title("Test 4: IV mean reversion (all tickers pooled)"); plt.show()

## Test 5: Put credit spreads: breach rate and P&L
For each ticker, every `ENTRY_STEP` trading days: sell a put at about `TARGET_DELTA` using that day's IV, buy a put a fixed width below, hold to expiry (21 rows later), and settle on the actual price. **Model assumptions (all of which flatter or hurt the result):** one flat IV for both strikes (real puts have skew, which raises the credit and the price of the long leg), mid price minus a small slippage, entry fees, no early exit, no assignment effects. This tests the *signal*, not tastytrade's real fills.

Key comparison: the strike's nominal chance of being breached (N(-d2)) versus how often it was.

In [ ]:
def bs_put(S, K, T, sig, r=RF):
    d1 = (np.log(S / K) + (r + sig ** 2 / 2) * T) / (sig * np.sqrt(T)); d2 = d1 - sig * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def short_strike(S, T, sig, delta=TARGET_DELTA, r=RF):
    d1 = norm.ppf(1 - delta)                       # put delta = N(d1) - 1 = -delta
    return S * np.exp((r + sig ** 2 / 2) * T - d1 * sig * np.sqrt(T))

def spread_width(S):
    return float(min(5, max(1, round(0.01 * S))))   # about 1% of spot, between $1 and $5

T_ = DTE_DAYS / 365
trades = []
for t, g in F.groupby("ticker"):
    g = g.sort_values("date").reset_index(drop=True)
    for i in range(0, len(g) - HORIZON, ENTRY_STEP):
        S, sig = g.at[i, "price"], g.at[i, "iv"]
        if not (np.isfinite(S) and np.isfinite(sig) and sig > 0.03): continue
        ST = g.at[i + HORIZON, "price"]; Ks = short_strike(S, T_, sig); w = spread_width(S); Kl = Ks - w
        credit = bs_put(S, Ks, T_, sig) - bs_put(S, Kl, T_, sig) - SLIPPAGE
        if credit <= 0.02 or credit >= w: continue
        d2 = (np.log(S / Ks) + (RF - sig ** 2 / 2) * T_) / (sig * np.sqrt(T_))
        payoff = max(Ks - ST, 0) - max(Kl - ST, 0)
        fees = 2 * FEE_PER_LEG
        pnl = (credit - payoff) * 100 - fees; maxloss = (w - credit) * 100 + fees
        trades.append(dict(ticker=t, date=g.at[i, "date"], S=S, iv=sig, width=w, credit=credit, nominal_breach=norm.cdf(-d2),
                           breached=ST < Ks, pnl=pnl, maxloss=maxloss, ror=pnl / maxloss, ivr=g.at[i, "ivr"],
                           ratio21=g.at[i, "iv"] / g.at[i, "rv21"] if g.at[i, "rv21"] > 0 else np.nan, rv21=g.at[i, "rv21"], garch=g.at[i, "garch"]))
TR = pd.DataFrame(trades)
blk = max(1, int(np.ceil(HORIZON / ENTRY_STEP)))
print("simulated trades:", len(TR), "| tickers:", TR.ticker.nunique(), "| effective independent trades is roughly", int(len(TR) * ENTRY_STEP / HORIZON), "(entries overlap)")

def summarize(df, label):
    pt, lo, hi = mbb_mean_ci([g.pnl.to_numpy() for _, g in df.groupby("ticker")], block=blk, n_boot=800)
    wins = (df.pnl > 0).mean(); avg_w, avg_l = df.pnl[df.pnl > 0].mean(), -df.pnl[df.pnl <= 0].mean()
    return dict(set=label, n=len(df), nominal_breach_pct=100 * df.nominal_breach.mean(), actual_breach_pct=100 * df.breached.mean(),
                win_pct=100 * wins, breakeven_win_pct=100 * (avg_l / (avg_w + avg_l)) if avg_w == avg_w and avg_l == avg_l else np.nan,
                mean_pnl_usd=pt, ci_lo=lo, ci_hi=hi, avg_win=avg_w, avg_loss=-avg_l, worst=df.pnl.min())
res5 = pd.DataFrame([summarize(TR, "all entries")]).set_index("set"); display(res5)
SUMMARY["Spread: actual vs nominal breach %"] = f"{res5.actual_breach_pct.iloc[0]:.1f} vs {res5.nominal_breach_pct.iloc[0]:.1f}"
SUMMARY["Spread: mean P&L per 1-lot ($)"] = f"{res5.mean_pnl_usd.iloc[0]:.1f} [{res5.ci_lo.iloc[0]:.1f}, {res5.ci_hi.iloc[0]:.1f}]"

bt = TR.groupby("ticker").apply(lambda g: pd.Series(dict(n=len(g), nominal_breach_pct=100 * g.nominal_breach.mean(), actual_breach_pct=100 * g.breached.mean(),
                                                         win_pct=100 * (g.pnl > 0).mean(), mean_pnl=g.pnl.mean(), typical_maxloss=g.maxloss.median())))
display(bt)
fig, ax = plt.subplots(); x = np.arange(len(bt)); ax.bar(x - .2, bt.nominal_breach_pct, .4, label="nominal (model)"); ax.bar(x + .2, bt.actual_breach_pct, .4, label="actual")
ax.set_xticks(x); ax.set_xticklabels(bt.index, rotation=45); ax.set_ylabel("% of trades with the short strike breached"); ax.legend(); ax.set_title("Test 5: breach rate vs the model's probability"); plt.show()

print("\nDoes the screener's filter help? Mean P&L per 1-lot by signal (95% block-bootstrap interval):")
f = TR.dropna(subset=["ratio21", "ivr"])
sig_tab = pd.DataFrame([summarize(f[f.ratio21 >= RICH_AT], "IV/HV21 rich"), summarize(f[f.ratio21 < RICH_AT], "IV/HV21 not rich"),
                        summarize(f[f.ivr >= IVR_HI], "IV rank >= %d" % IVR_HI), summarize(f[f.ivr < IVR_HI], "IV rank < %d" % IVR_HI),
                        summarize(f[(f.ratio21 >= RICH_AT) & (f.ivr >= IVR_HI)], "rich AND high rank"),
                        summarize(f[(f.ratio21 < RICH_AT) | (f.ivr < IVR_HI)], "anything else")]).set_index("set")
display(sig_tab[["n", "actual_breach_pct", "win_pct", "mean_pnl_usd", "ci_lo", "ci_hi", "worst"]])

print("\nCan a $%d account hold a 1-lot? Typical max loss per spread:" % ACCOUNT)
fit = TR.groupby("ticker").agg(spot=("S", "median"), width=("width", "median"), max_loss_usd=("maxloss", "median"))
fit["pct_of_account"] = 100 * fit.max_loss_usd / ACCOUNT; fit["fits"] = fit.pct_of_account <= 100 * MAX_RISK_PCT
display(fit)

## Test 6: Fat tails
The GARCH video: a normal distribution with constant vol understates risk, and ignoring volatility clustering is worse. Here: one-day 5% value-at-risk exceedance rates using an expanding-window standard deviation (constant vol) versus EWMA vol, both assuming normality. A well-calibrated 5% VaR should be breached about 5% of the time.

In [ ]:
rows = []
for t, g in F.groupby("ticker"):
    r = g["ret"].dropna().reset_index(drop=True)
    if len(r) < 150: continue
    sd_const = r.expanding(60).std().shift(1)
    sd_ewma = np.sqrt((r ** 2).ewm(alpha=0.06, adjust=False).mean()).shift(1)
    mask = sd_const.notna() & sd_ewma.notna()
    q = norm.ppf(0.05)
    rows.append(dict(ticker=t, excess_kurtosis=r.kurt(), exceed_const_vol_pct=100 * (r[mask] < q * sd_const[mask]).mean(),
                     exceed_ewma_pct=100 * (r[mask] < q * sd_ewma[mask]).mean(), worst_day_in_sigmas=(r.min() / r.std())))
ft = pd.DataFrame(rows).set_index("ticker"); display(ft); display(ft.mean().to_frame("mean across tickers").T)
SUMMARY["5% VaR exceedance: constant vol / EWMA (%)"] = f"{ft.exceed_const_vol_pct.mean():.1f} / {ft.exceed_ewma_pct.mean():.1f}"
print("A rate well above 5% means the normal model understates the downside. Spreads cap the loss, but a breach is more likely than a normal-based probability of profit says.")

## Test 7: Sizing: Kelly and a $3,000 account
Volatility drag in practice: wealth compounds geometrically, so the bet size that matters is the one that maximizes long-run growth, not average profit. Using the simulated trades' **return on risk** (P&L divided by max loss):

1. The Kelly fraction (and how fragile it is to a small overestimate of the edge).
2. A Monte Carlo of a year of trades at different risk fractions. Trades are drawn independently, which **understates** how losses cluster in a selloff, so a stressed version (losses 1.5x larger) is shown too.
3. Fractional contracts are allowed here; with a $3,000 account you can only trade whole contracts, so see the affordability table in Test 5.

In [ ]:
ror = TR.ror.dropna().to_numpy()
grid = np.linspace(0, 0.6, 121)
def growth(f, x): return np.mean(np.log1p(f * x))
kelly_rows = []
for shave in (0.0, 0.01, 0.02, 0.03, 0.05):
    x = np.clip(ror - shave, -1, None)
    gs = np.array([growth(f, x) for f in grid]); fstar = grid[gs.argmax()]
    kelly_rows.append(dict(edge_overstated_by=f"{int(shave * 100)}% of risk", kelly_fraction=fstar, quarter_kelly=fstar / 4, growth_at_kelly_per_trade=gs.max()))
display(pd.DataFrame(kelly_rows).set_index("edge_overstated_by"))
SUMMARY["Kelly fraction (no shave / 2% shave)"] = f"{kelly_rows[0]['kelly_fraction']:.2f} / {kelly_rows[2]['kelly_fraction']:.2f}"

rng = np.random.RandomState(SEED)
N_TRADES, N_PATHS = 24, 4000
def simulate(fr, loss_mult=1.0):
    draws = rng.choice(ror, size=(N_PATHS, N_TRADES))
    draws = np.where(draws < 0, draws * loss_mult, draws).clip(-1, None)
    eq = ACCOUNT * np.cumprod(1 + fr * draws, axis=1)
    peak = np.maximum.accumulate(np.hstack([np.full((N_PATHS, 1), ACCOUNT), eq]), axis=1)[:, 1:]
    dd = (1 - eq / peak).max(axis=1)
    return eq[:, -1], dd
rows = []
for lm, lab in ((1.0, "as simulated"), (1.5, "stress: losses 1.5x")):
    for fr in (0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30):
        end, dd = simulate(fr, lm)
        rows.append(dict(case=lab, risk_per_trade=f"{int(fr * 100)}%", median_end=np.median(end), p5_end=np.percentile(end, 5),
                         prob_below_start=100 * (end < ACCOUNT).mean(), prob_drawdown_over_30pct=100 * (dd > 0.30).mean(), prob_lose_half=100 * (end < ACCOUNT / 2).mean()))
mc = pd.DataFrame(rows).set_index(["case", "risk_per_trade"]); display(mc)
fig, ax = plt.subplots()
for fr in (0.03, 0.10, 0.30):
    draws = rng.choice(ror, size=(1500, N_TRADES)); eq = ACCOUNT * np.cumprod(1 + fr * draws, axis=1)
    ax.plot(np.arange(1, N_TRADES + 1), np.median(eq, axis=0), label=f"median, risk {int(fr * 100)}%/trade")
    ax.fill_between(np.arange(1, N_TRADES + 1), np.percentile(eq, 5, axis=0), np.percentile(eq, 95, axis=0), alpha=.12)
ax.axhline(ACCOUNT, color="k", lw=.7); ax.set_xlabel("trade number"); ax.set_ylabel("account value ($)"); ax.legend(); ax.set_title("Test 7: account paths (5th to 95th percentile bands)"); plt.show()

## Summary

In [ ]:
print("DATA:", "SYNTHETIC (code check only)" if IS_SYNTH else "Unusual Whales live history", "\n")
for k, v in SUMMARY.items(): print(f"{k:60s} {v}")
print("""
How to read this:
- Test 1/3 say whether selling premium has an edge and whether the screener's labels sort it. A bootstrap interval that includes zero means 'not shown', not 'no edge'.
- Test 2 picks the realized-vol denominator for IV/HV. Close scores are ties; 2 years cannot separate them.
- Test 4 is the video's mean-reversion idea tested out of sample, which is the part the video skips.
- Test 5 is a model of spread P&L, not real fills. Add skew, real bid/ask and early exits before trusting a dollar figure.
- Test 7: if the edge is overstated by even a few percent of risk, the sensible bet size falls sharply. Size small.
Next: rerun with a larger universe, add the 10-year Yahoo price history for breach rates, then forward-test with logged trades.
""")